In [ ]:
import os
import random
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from PIL import Image
from pathlib import Path
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms, models
from torchvision.models import (
    googlenet, GoogLeNet_Weights,
    vgg16, VGG16_Weights
)

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    confusion_matrix, classification_report,
    ConfusionMatrixDisplay
)
from tqdm import tqdm

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# SECTION 1 — CONFIGURATION  (⚠️ CHANGE DATA_DIR TO YOUR PATH)
# ─────────────────────────────────────────────────────────────────────────────
DATA_DIR = '/kaggle/input/datasets/mohammadamireshraghi/blood-cell-cancer-all-4class/Blood cell Cancer [ALL]'

# ── Training settings ──────────────────────────────────────────────────────
IMAGE_SIZE   = 224        # All models expect 224×224
BATCH_SIZE   = 32         # Reduce to 16 if you get out-of-memory errors
NUM_EPOCHS   = 15         # Epochs per model (increase for better results)
LEARNING_RATE = 0.001
SEED         = 42         # For reproducibility

# ── Class names (must match folder names exactly) ─────────────────────────
CLASS_NAMES = ['Benign', '[Malignant] early Pre-B', '[Malignant] Pre-B', '[Malignant] Pro-B']
NUM_CLASSES  = len(CLASS_NAMES)

# ── Output directory for saving plots ─────────────────────────────────────
OUTPUT_DIR = './outputs'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ── Device: automatically uses GPU if available ───────────────────────────
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"✅ Using device: {DEVICE}")
if DEVICE.type == 'cuda':
    print(f"   GPU: {torch.cuda.get_device_name(0)}")

# ── Reproducibility ───────────────────────────────────────────────────────
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True

set_seed(SEED)

In [ ]:
print("\n" + "="*60)
print("📊 SECTION 2: DATA EXPLORATION")
print("="*60)

def collect_all_image_paths(data_dir):
    """Walk through dataset folders and collect all image paths + labels."""
    image_paths = []
    labels      = []

    for class_idx, class_name in enumerate(CLASS_NAMES):
        class_dir = Path(data_dir) / class_name
        if not class_dir.exists():
            print(f"⚠️  Folder not found: {class_dir}")
            continue
        images = list(class_dir.glob('*.jpg')) + \
                 list(class_dir.glob('*.JPG')) + \
                 list(class_dir.glob('*.png')) + \
                 list(class_dir.glob('*.PNG'))
        image_paths.extend(images)
        labels.extend([class_idx] * len(images))
        print(f"   Class '{class_name}': {len(images)} images")

    print(f"\n   Total images found: {len(image_paths)}")
    return image_paths, labels
image_paths, labels = collect_all_image_paths(DATA_DIR)

# ── VIZ 1: Class Distribution Bar Chart ──────────────────────────────────
def plot_class_distribution(labels, class_names, save_path):
    counts = Counter(labels)
    counts_sorted = [counts[i] for i in range(len(class_names))]

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle('Class Distribution in Dataset', fontsize=15, fontweight='bold')

    # Bar chart
    colors = ['#2ecc71', '#e74c3c', '#e67e22', '#9b59b6']
    bars = axes[0].bar(range(len(class_names)), counts_sorted, color=colors, edgecolor='black', linewidth=0.8)
    axes[0].set_xticks(range(len(class_names)))
    axes[0].set_xticklabels([c.replace('[Malignant] ', '') for c in class_names], rotation=15, ha='right')
    axes[0].set_ylabel('Number of Images')
    axes[0].set_title('Images per Class')
    for bar, count in zip(bars, counts_sorted):
        axes[0].text(bar.get_x() + bar.get_width()/2., bar.get_height() + 10,
                     str(count), ha='center', va='bottom', fontweight='bold')

    # Pie chart
    axes[1].pie(counts_sorted, labels=[c.replace('[Malignant] ', '') for c in class_names],
                colors=colors, autopct='%1.1f%%', startangle=140,
                wedgeprops=dict(edgecolor='white', linewidth=2))
    axes[1].set_title('Class Proportion')
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f"   💾 Saved: {save_path}")

plot_class_distribution(labels, CLASS_NAMES, f'{OUTPUT_DIR}/viz1_class_distribution.png')

# ── VIZ 2: Sample Images Grid (4 per class) ───────────────────────────────
def plot_sample_images(image_paths, labels, class_names, save_path, samples_per_class=4):
    fig, axes = plt.subplots(len(class_names), samples_per_class,
                              figsize=(samples_per_class * 3, len(class_names) * 3))
    fig.suptitle('Sample Images per Class (RAW — Before Any Processing)',
                 fontsize=13, fontweight='bold', y=1.01)

    colors = ['#2ecc71', '#e74c3c', '#e67e22', '#9b59b6']

    for class_idx, class_name in enumerate(class_names):
        # Get all paths for this class
        class_paths = [p for p, l in zip(image_paths, labels) if l == class_idx]
        sampled = random.sample(class_paths, min(samples_per_class, len(class_paths)))

        for col, img_path in enumerate(sampled):
            img = Image.open(img_path).convert('RGB')
            axes[class_idx][col].imshow(img)
            axes[class_idx][col].axis('off')
            if col == 0:
                short_name = class_name.replace('[Malignant] ', 'Malig.\n')
                axes[class_idx][col].set_ylabel(short_name, fontsize=10,
                                                 rotation=0, labelpad=60,
                                                 va='center',
                                                 color=colors[class_idx],
                                                 fontweight='bold')
                axes[class_idx][col].yaxis.set_label_coords(-0.5, 0.5)

    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f"   💾 Saved: {save_path}")

plot_sample_images(image_paths, labels, CLASS_NAMES,
                   f'{OUTPUT_DIR}/viz2_sample_images_raw.png')

# ── VIZ 3: Image Size Distribution ───────────────────────────────────────
def plot_image_size_distribution(image_paths, save_path, sample_n=200):
    """Check if all images are the same size or vary."""
    print("   Checking image sizes (sampling 200 images)...")
    sampled_paths = random.sample(image_paths, min(sample_n, len(image_paths)))
    widths, heights = [], []

    for p in sampled_paths:
        try:
            w, h = Image.open(p).size
            widths.append(w)
            heights.append(h)
        except Exception:
            pass
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    fig.suptitle('Image Size Distribution (sample of 200 images)', fontsize=13, fontweight='bold')

    axes[0].hist(widths, bins=20, color='#3498db', edgecolor='black')
    axes[0].set_xlabel('Width (pixels)')
    axes[0].set_ylabel('Count')
    axes[0].set_title(f'Width  |  Mean: {np.mean(widths):.0f}px')

    axes[1].hist(heights, bins=20, color='#e74c3c', edgecolor='black')
    axes[1].set_xlabel('Height (pixels)')
    axes[1].set_ylabel('Count')
    axes[1].set_title(f'Height  |  Mean: {np.mean(heights):.0f}px')

    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f"   💾 Saved: {save_path}")
    print(f"   Width  range: {min(widths)} – {max(widths)} px")
    print(f"   Height range: {min(heights)} – {max(heights)} px")

plot_image_size_distribution(image_paths, f'{OUTPUT_DIR}/viz3_image_sizes.png')

# ── VIZ 4: Pixel Intensity Distribution per class ─────────────────────────
def plot_pixel_intensity(image_paths, labels, class_names, save_path, sample_n=50):
    """Show the average brightness distribution for each class."""
    print("   Computing pixel intensities...")
    fig, ax = plt.subplots(figsize=(10, 5))
    colors = ['#2ecc71', '#e74c3c', '#e67e22', '#9b59b6']
    for class_idx, (class_name, color) in enumerate(zip(class_names, colors)):
        class_paths = [p for p, l in zip(image_paths, labels) if l == class_idx]
        sampled = random.sample(class_paths, min(sample_n, len(class_paths)))
        all_pixels = []
        for p in sampled:
            img = np.array(Image.open(p).convert('L').resize((64, 64)))  # grayscale
            all_pixels.extend(img.flatten().tolist())

        ax.hist(all_pixels, bins=50, alpha=0.5, color=color, density=True,
                label=class_name.replace('[Malignant] ', 'Malig. '))

    ax.set_xlabel('Pixel Intensity (0=Black, 255=White)')
    ax.set_ylabel('Density')
    ax.set_title('Pixel Intensity Distribution per Class (Raw Images)')
    ax.legend()
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f"   💾 Saved: {save_path}")
  
plot_pixel_intensity(image_paths, labels, CLASS_NAMES,
                     f'{OUTPUT_DIR}/viz4_pixel_intensity_raw.png')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# SECTION 2 — DATA EXPLORATION & VISUALIZATIONS (BEFORE PROCESSING)
# ─────────────────────────────────────────────────────────────────────────────

print("\n" + "="*60)
print("📊 SECTION 2: DATA EXPLORATION")
print("="*60)

def collect_all_image_paths(data_dir):
    """Walk through dataset folders and collect all image paths + labels."""
    image_paths = []
    labels      = []

    for class_idx, class_name in enumerate(CLASS_NAMES):
        class_dir = Path(data_dir) / class_name
        if not class_dir.exists():
            print(f"⚠️  Folder not found: {class_dir}")
            continue
        images = list(class_dir.glob('*.jpg')) + \
                 list(class_dir.glob('*.JPG')) + \
                 list(class_dir.glob('*.png')) + \
                 list(class_dir.glob('*.PNG'))
        image_paths.extend(images)
        labels.extend([class_idx] * len(images))
        print(f"   Class '{class_name}': {len(images)} images")

    print(f"\n   Total images found: {len(image_paths)}")
    return image_paths, labels

image_paths, labels = collect_all_image_paths(DATA_DIR)

# ── VIZ 1: Class Distribution Bar Chart ──────────────────────────────────
def plot_class_distribution(labels, class_names, save_path):
    counts = Counter(labels)
    counts_sorted = [counts[i] for i in range(len(class_names))]

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle('Class Distribution in Dataset', fontsize=15, fontweight='bold')

    # Bar chart
    colors = ['#2ecc71', '#e74c3c', '#e67e22', '#9b59b6']
    bars = axes[0].bar(range(len(class_names)), counts_sorted, color=colors, edgecolor='black', linewidth=0.8)
    axes[0].set_xticks(range(len(class_names)))
    axes[0].set_xticklabels([c.replace('[Malignant] ', '') for c in class_names], rotation=15, ha='right')
    axes[0].set_ylabel('Number of Images')
    axes[0].set_title('Images per Class')
    for bar, count in zip(bars, counts_sorted):
        axes[0].text(bar.get_x() + bar.get_width()/2., bar.get_height() + 10,
                     str(count), ha='center', va='bottom', fontweight='bold')

    # Pie chart
    axes[1].pie(counts_sorted, labels=[c.replace('[Malignant] ', '') for c in class_names],
                colors=colors, autopct='%1.1f%%', startangle=140,
                wedgeprops=dict(edgecolor='white', linewidth=2))
    axes[1].set_title('Class Proportion')

    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f"   💾 Saved: {save_path}")

plot_class_distribution(labels, CLASS_NAMES, f'{OUTPUT_DIR}/viz1_class_distribution.png')

# ── VIZ 2: Sample Images Grid (4 per class) ───────────────────────────────
def plot_sample_images(image_paths, labels, class_names, save_path, samples_per_class=4):
    fig, axes = plt.subplots(len(class_names), samples_per_class,
                              figsize=(samples_per_class * 3, len(class_names) * 3))
    fig.suptitle('Sample Images per Class (RAW — Before Any Processing)',
                 fontsize=13, fontweight='bold', y=1.01)

    colors = ['#2ecc71', '#e74c3c', '#e67e22', '#9b59b6']

    for class_idx, class_name in enumerate(class_names):
        # Get all paths for this class
        class_paths = [p for p, l in zip(image_paths, labels) if l == class_idx]
        sampled = random.sample(class_paths, min(samples_per_class, len(class_paths)))

        for col, img_path in enumerate(sampled):
            img = Image.open(img_path).convert('RGB')
            axes[class_idx][col].imshow(img)
            axes[class_idx][col].axis('off')
            if col == 0:
                short_name = class_name.replace('[Malignant] ', 'Malig.\n')
                axes[class_idx][col].set_ylabel(short_name, fontsize=10,
                                                 rotation=0, labelpad=60,
                                                 va='center',
                                                 color=colors[class_idx],
                                                 fontweight='bold')
                axes[class_idx][col].yaxis.set_label_coords(-0.5, 0.5)

    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f"   💾 Saved: {save_path}")

plot_sample_images(image_paths, labels, CLASS_NAMES,
                   f'{OUTPUT_DIR}/viz2_sample_images_raw.png')

# ── VIZ 3: Image Size Distribution ───────────────────────────────────────
def plot_image_size_distribution(image_paths, save_path, sample_n=200):
    """Check if all images are the same size or vary."""
    print("   Checking image sizes (sampling 200 images)...")
    sampled_paths = random.sample(image_paths, min(sample_n, len(image_paths)))
    widths, heights = [], []

    for p in sampled_paths:
        try:
            w, h = Image.open(p).size
            widths.append(w)
            heights.append(h)
        except Exception:
            pass

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    fig.suptitle('Image Size Distribution (sample of 200 images)', fontsize=13, fontweight='bold')

    axes[0].hist(widths, bins=20, color='#3498db', edgecolor='black')
    axes[0].set_xlabel('Width (pixels)')
    axes[0].set_ylabel('Count')
    axes[0].set_title(f'Width  |  Mean: {np.mean(widths):.0f}px')

    axes[1].hist(heights, bins=20, color='#e74c3c', edgecolor='black')
    axes[1].set_xlabel('Height (pixels)')
    axes[1].set_ylabel('Count')
    axes[1].set_title(f'Height  |  Mean: {np.mean(heights):.0f}px')

    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f"   💾 Saved: {save_path}")
    print(f"   Width  range: {min(widths)} – {max(widths)} px")
    print(f"   Height range: {min(heights)} – {max(heights)} px")

plot_image_size_distribution(image_paths, f'{OUTPUT_DIR}/viz3_image_sizes.png')

# ── VIZ 4: Pixel Intensity Distribution per class ─────────────────────────
def plot_pixel_intensity(image_paths, labels, class_names, save_path, sample_n=50):
    """Show the average brightness distribution for each class."""
    print("   Computing pixel intensities...")
    fig, ax = plt.subplots(figsize=(10, 5))
    colors = ['#2ecc71', '#e74c3c', '#e67e22', '#9b59b6']

    for class_idx, (class_name, color) in enumerate(zip(class_names, colors)):
        class_paths = [p for p, l in zip(image_paths, labels) if l == class_idx]
        sampled = random.sample(class_paths, min(sample_n, len(class_paths)))
        all_pixels = []
        for p in sampled:
            img = np.array(Image.open(p).convert('L').resize((64, 64)))  # grayscale
            all_pixels.extend(img.flatten().tolist())

        ax.hist(all_pixels, bins=50, alpha=0.5, color=color, density=True,
                label=class_name.replace('[Malignant] ', 'Malig. '))

    ax.set_xlabel('Pixel Intensity (0=Black, 255=White)')
    ax.set_ylabel('Density')
    ax.set_title('Pixel Intensity Distribution per Class (Raw Images)')
    ax.legend()
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f"   💾 Saved: {save_path}")

plot_pixel_intensity(image_paths, labels, CLASS_NAMES,
                     f'{OUTPUT_DIR}/viz4_pixel_intensity_raw.png')



In [ ]:


# ─────────────────────────────────────────────────────────────────────────────
# SECTION 3 — DATASET SPLIT (by index, stratified)
# ─────────────────────────────────────────────────────────────────────────────

print("\n" + "="*60)
print("✂️  SECTION 3: TRAIN / VAL / TEST SPLIT")
print("="*60)

# Split: 70% train, 15% val, 15% test — stratified so each split has same class ratios
indices = list(range(len(image_paths)))

train_idx, temp_idx = train_test_split(
    indices, test_size=0.30, stratify=labels, random_state=SEED
)
val_idx, test_idx = train_test_split(
    temp_idx, test_size=0.50,
    stratify=[labels[i] for i in temp_idx],
    random_state=SEED
)

print(f"   Train : {len(train_idx)} images ({len(train_idx)/len(indices)*100:.1f}%)")
print(f"   Val   : {len(val_idx)} images ({len(val_idx)/len(indices)*100:.1f}%)")
print(f"   Test  : {len(test_idx)} images ({len(test_idx)/len(indices)*100:.1f}%)")



In [ ]:

# ─────────────────────────────────────────────────────────────────────────────
# SECTION 4 — TRANSFORMS (PREPROCESSING + AUGMENTATION)
# ─────────────────────────────────────────────────────────────────────────────

print("\n" + "="*60)
print("🔧 SECTION 4: PREPROCESSING & AUGMENTATION PIPELINE")
print("="*60)

# ── ImageNet mean/std — used because our pretrained models learned these ──
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

# TRAINING transforms: includes augmentation to prevent overfitting
train_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),    # Make all images same size
    transforms.RandomHorizontalFlip(p=0.5),         # Flip left-right (cells have no orientation)
    transforms.RandomVerticalFlip(p=0.5),           # Flip up-down (same reason)
    transforms.RandomRotation(degrees=360),         # Rotate any angle (cells look same rotated)
    transforms.ColorJitter(                         # Simulate staining variation between batches
        brightness=0.2,
        contrast=0.2,
        saturation=0.1
    ),
    transforms.ToTensor(),                          # Convert PIL image → PyTorch tensor [0,1]
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD)  # Normalize to ImageNet stats
])

# VALIDATION/TEST transforms: NO augmentation — just resize + normalize
val_test_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD)
])

# ── VIZ 5: Before vs After Augmentation ──────────────────────────────────
def plot_augmentation_examples(image_paths, labels, class_names, save_path):
    """Show 1 raw image vs 4 augmented versions side by side."""
    fig, axes = plt.subplots(len(class_names), 5, figsize=(16, len(class_names) * 3.2))
    fig.suptitle('Before vs After Augmentation (1 raw + 4 augmented versions)',
                 fontsize=13, fontweight='bold')

    aug_only = transforms.Compose([
        transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomVerticalFlip(p=0.5),
        transforms.RandomRotation(degrees=360),
        transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1),
    ])

    for row, class_idx in enumerate(range(len(class_names))):
        class_paths = [p for p, l in zip(image_paths, labels) if l == class_idx]
        img_path    = random.choice(class_paths)
        raw_img     = Image.open(img_path).convert('RGB').resize((IMAGE_SIZE, IMAGE_SIZE))

        # Column 0: raw
        axes[row][0].imshow(raw_img)
        axes[row][0].set_title('RAW', fontsize=9, color='blue', fontweight='bold')
        axes[row][0].axis('off')
        if row == 0:
            axes[row][0].set_title('ORIGINAL\n(resized)', fontsize=9, color='blue', fontweight='bold')

        # Columns 1–4: augmented versions
        for col in range(1, 5):
            aug_img = aug_only(raw_img)
            axes[row][col].imshow(aug_img)
            axes[row][col].set_title(f'Aug #{col}', fontsize=9)
            axes[row][col].axis('off')

        # Row label
        short = class_names[class_idx].replace('[Malignant] ', 'M.')
        axes[row][0].set_ylabel(short, fontsize=9, rotation=0,
                                 labelpad=55, va='center', fontweight='bold')

    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f"   💾 Saved: {save_path}")

plot_augmentation_examples(image_paths, labels, CLASS_NAMES,
                           f'{OUTPUT_DIR}/viz5_augmentation_examples.png')

# ── VIZ 6: Pixel value distribution before vs after normalization ─────────
def plot_normalization_effect(image_paths, save_path):
    """Show pixel histograms before and after normalization."""
    sample_path = random.choice(image_paths)
    raw = Image.open(sample_path).convert('RGB').resize((IMAGE_SIZE, IMAGE_SIZE))
    raw_np = np.array(raw) / 255.0  # 0–1 range before normalization

    to_tensor_norm = transforms.Compose([
        transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD)
    ])
    norm_tensor = to_tensor_norm(Image.open(sample_path).convert('RGB'))
    norm_np = norm_tensor.permute(1, 2, 0).numpy()  # C,H,W → H,W,C

    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    fig.suptitle('Effect of Normalization on Pixel Values', fontsize=13, fontweight='bold')

    axes[0].imshow(raw)
    axes[0].set_title('Original Image')
    axes[0].axis('off')

    colors_rgb = ['red', 'green', 'blue']
    for ch, col in enumerate(colors_rgb):
        axes[1].hist(raw_np[:, :, ch].flatten(), bins=50, alpha=0.5,
                     color=col, label=f'Ch {ch}', density=True)
    axes[1].set_title('Before Normalization\n(pixel values 0.0 – 1.0)')
    axes[1].set_xlabel('Pixel Value')
    axes[1].legend()

    for ch, col in enumerate(colors_rgb):
        axes[2].hist(norm_np[:, :, ch].flatten(), bins=50, alpha=0.5,
                     color=col, label=f'Ch {ch}', density=True)
    axes[2].set_title('After Normalization\n(centered around 0, spread ~1)')
    axes[2].set_xlabel('Pixel Value')
    axes[2].legend()

    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f"   💾 Saved: {save_path}")

plot_normalization_effect(image_paths, f'{OUTPUT_DIR}/viz6_normalization_effect.png')



In [ ]:

# ─────────────────────────────────────────────────────────────────────────────
# SECTION 5 — CUSTOM DATASET CLASS & DATALOADERS
# ─────────────────────────────────────────────────────────────────────────────

print("\n" + "="*60)
print("📦 SECTION 5: BUILDING DATALOADERS")
print("="*60)

class ALLDataset(Dataset):
    """
    Custom PyTorch Dataset for the ALL Blood Cancer images.
    Think of this as a "smart list" that:
    1. Holds the image paths and labels
    2. Opens and transforms each image when the DataLoader asks for it
    """
    def __init__(self, image_paths, labels, transform=None):
        self.image_paths = image_paths
        self.labels      = labels
        self.transform   = transform

    def __len__(self):
        # How many images are in this dataset?
        return len(self.image_paths)

    def __getitem__(self, idx):
        # Load one image by index
        img   = Image.open(self.image_paths[idx]).convert('RGB')
        label = self.labels[idx]
        if self.transform:
            img = self.transform(img)
        return img, label


# Build path/label lists for each split
train_paths  = [image_paths[i] for i in train_idx]
train_labels = [labels[i] for i in train_idx]
val_paths    = [image_paths[i] for i in val_idx]
val_labels   = [labels[i] for i in val_idx]
test_paths   = [image_paths[i] for i in test_idx]
test_labels  = [labels[i] for i in test_idx]

# ── Weighted Sampler: fix class imbalance ─────────────────────────────────
# Classes with fewer images get sampled more often during training
train_counts = Counter(train_labels)
class_weights = {cls: 1.0 / count for cls, count in train_counts.items()}
sample_weights = [class_weights[l] for l in train_labels]
sampler = WeightedRandomSampler(
    weights=sample_weights,
    num_samples=len(sample_weights),
    replacement=True
)

# Create Dataset objects
train_dataset = ALLDataset(train_paths, train_labels, transform=train_transform)
val_dataset   = ALLDataset(val_paths,   val_labels,   transform=val_test_transform)
test_dataset  = ALLDataset(test_paths,  test_labels,  transform=val_test_transform)

# Create DataLoaders (these feed batches of images to the model)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE,
                          sampler=sampler, num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE,
                          shuffle=False,  num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE,
                          shuffle=False,  num_workers=2, pin_memory=True)

print(f"   Train batches: {len(train_loader)}")
print(f"   Val   batches: {len(val_loader)}")
print(f"   Test  batches: {len(test_loader)}")



In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# SECTION 6 — MODEL ARCHITECTURES
# ─────────────────────────────────────────────────────────────────────────────

print("\n" + "="*60)
print("🏗️  SECTION 6: MODEL DEFINITIONS")
print("="*60)

# ── MODEL 1: Simple CNN ───────────────────────────────────────────────────
class SimpleCNN(nn.Module):
    """
    A basic CNN built from scratch.
    Architecture:
      Conv(3→32) → BN → ReLU → Pool
      Conv(32→64) → BN → ReLU → Pool
      Conv(64→128) → BN → ReLU → Pool
      Conv(128→256) → BN → ReLU → Pool
      Flatten → FC(256*14*14 → 512) → Dropout → FC(512 → 4)
    """
    def __init__(self, num_classes=4):
        super(SimpleCNN, self).__init__()

        self.features = nn.Sequential(
            # Block 1
            nn.Conv2d(3, 32, kernel_size=3, padding=1),   # input: 3-channel RGB
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),                            # 224 → 112

            # Block 2
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),                            # 112 → 56

            # Block 3
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),                            # 56 → 28

            # Block 4
            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),                            # 28 → 14
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256 * 14 * 14, 512),
            nn.ReLU(inplace=True),
            nn.Dropout(p=0.5),                            # Dropout prevents overfitting
            nn.Linear(512, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x


# ── MODEL 2: GoogLeNet from Scratch ───────────────────────────────────────
def build_googlenet_scratch(num_classes=4):
    """
    GoogLeNet architecture, trained from random weights.
    aux_logits=True keeps auxiliary outputs active (helps gradient flow).
    The training loop handles their outputs automatically.
    """
    model = googlenet(weights=None, num_classes=num_classes, aux_logits=True)
    return model


# ── MODEL 3: VGG16 from Scratch ───────────────────────────────────────────
def build_vgg16_scratch(num_classes=4):
    """VGG16 architecture, trained from random weights."""
    model = vgg16(weights=None)
    # Replace the final classifier layer to output 4 classes instead of 1000
    model.classifier[6] = nn.Linear(4096, num_classes)
    return model


# ── MODEL 4: GoogLeNet with Transfer Learning ─────────────────────────────
def build_googlenet_transfer(num_classes=4):
    """
    GoogLeNet pre-trained on ImageNet.
    Strategy: Freeze early layers (keep learned features), retrain the head.

    ⚠️  IMPORTANT: torchvision forces aux_logits=True when loading pretrained
    weights (it's baked into the checkpoint). We load with aux_logits=True,
    then DISABLE them after loading so training stays simple.
    """
    # Step 1: load pretrained weights — aux_logits MUST be True here
    model = googlenet(weights=GoogLeNet_Weights.IMAGENET1K_V1, aux_logits=True)

    # Step 2: immediately disable aux classifiers so forward() returns 1 tensor
    model.aux_logits = False
    model.aux1       = None
    model.aux2       = None

    # Step 3: Freeze ALL layers
    for param in model.parameters():
        param.requires_grad = False

    # Step 4: Unfreeze the last two Inception blocks so they adapt to our data
    for param in model.inception5a.parameters():
        param.requires_grad = True
    for param in model.inception5b.parameters():
        param.requires_grad = True

    # Step 5: Replace the final FC for 4-class output (always trainable)
    in_features = model.fc.in_features
    model.fc = nn.Sequential(
        nn.Linear(in_features, 256),
        nn.ReLU(),
        nn.Dropout(0.4),
        nn.Linear(256, num_classes)
    )
    return model


# ── MODEL 5: VGG16 with Transfer Learning ─────────────────────────────────
def build_vgg16_transfer(num_classes=4):
    """
    VGG16 pre-trained on ImageNet.
    Strategy: Freeze feature extractor, only train classifier layers.
    """
    model = vgg16(weights=VGG16_Weights.IMAGENET1K_V1)

    # Freeze all feature extraction layers
    for param in model.features.parameters():
        param.requires_grad = False

    # Replace classifier head
    model.classifier = nn.Sequential(
        nn.Linear(25088, 1024),
        nn.ReLU(inplace=True),
        nn.Dropout(0.5),
        nn.Linear(1024, 256),
        nn.ReLU(inplace=True),
        nn.Dropout(0.4),
        nn.Linear(256, num_classes)
    )
    return model


# Count trainable parameters (useful to compare model sizes)
def count_params(model):
    total     = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total, trainable

# Print parameter counts for all models
model_builders = {
    'SimpleCNN':            lambda: SimpleCNN(NUM_CLASSES),
    'GoogLeNet_Scratch':    lambda: build_googlenet_scratch(NUM_CLASSES),
    'VGG16_Scratch':        lambda: build_vgg16_scratch(NUM_CLASSES),
    'GoogLeNet_Transfer':   lambda: build_googlenet_transfer(NUM_CLASSES),
    'VGG16_Transfer':       lambda: build_vgg16_transfer(NUM_CLASSES),
}

print("\n   Model Parameter Summary:")
print(f"   {'Model':<25} {'Total Params':>15} {'Trainable':>15}")
print("   " + "-"*55)
for name, builder in model_builders.items():
    m = builder()
    total, trainable = count_params(m)
    print(f"   {name:<25} {total:>15,} {trainable:>15,}")
    del m

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# SECTION 7 — TRAINING & EVALUATION FUNCTIONS
# ─────────────────────────────────────────────────────────────────────────────

def get_logits_and_loss(model, images, labels_batch, criterion):
    """
    Helper that handles GoogLeNet aux_logits outputs gracefully.

    During TRAINING, GoogLeNet (scratch) returns a GoogLeNetOutputs named tuple:
        output.logits      -- main classifier output
        output.aux_logits  -- auxiliary outputs (help gradient flow in deep nets)
    Combined loss: total = main_loss + 0.3 * aux_loss  (standard GoogLeNet recipe)

    All other models return a plain tensor -- this function handles both cases.
    """
    outputs = model(images)

    if hasattr(outputs, 'logits'):          # GoogLeNet named-tuple output
        main_logits = outputs.logits
        loss = criterion(main_logits, labels_batch)
        if hasattr(outputs, 'aux_logits') and outputs.aux_logits is not None:
            aux = outputs.aux_logits
            if isinstance(aux, (list, tuple)):
                for aux_out in aux:
                    if aux_out is not None:
                        loss = loss + 0.3 * criterion(aux_out, labels_batch)
            else:
                loss = loss + 0.3 * criterion(aux, labels_batch)
    else:                                   # Every other model: plain tensor
        main_logits = outputs
        loss = criterion(main_logits, labels_batch)

    return main_logits, loss


def train_one_epoch(model, loader, criterion, optimizer):
    """Run one full pass through training data. Returns avg loss and accuracy."""
    model.train()
    running_loss, correct, total = 0.0, 0, 0

    for images, labels_batch in tqdm(loader, desc='  Training', leave=False):
        images       = images.to(DEVICE)
        labels_batch = labels_batch.to(DEVICE)

        optimizer.zero_grad()
        logits, loss = get_logits_and_loss(model, images, labels_batch, criterion)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        _, predicted  = logits.max(1)
        correct      += predicted.eq(labels_batch).sum().item()
        total        += labels_batch.size(0)

    return running_loss / total, 100. * correct / total


def evaluate(model, loader, criterion):
    """
    Evaluate on val or test set.
    Returns: loss, accuracy, list of predictions, list of true labels.
    In eval() mode GoogLeNet returns a plain tensor (aux automatically disabled).
    """
    model.eval()
    running_loss, correct, total = 0.0, 0, 0
    all_preds, all_labels = [], []

    with torch.no_grad():
        for images, labels_batch in loader:
            images       = images.to(DEVICE)
            labels_batch = labels_batch.to(DEVICE)

            outputs = model(images)
            # eval() mode: GoogLeNet returns plain tensor; handle both cases safely
            logits = outputs.logits if hasattr(outputs, 'logits') else outputs
            loss   = criterion(logits, labels_batch)

            running_loss += loss.item() * images.size(0)
            _, predicted  = logits.max(1)
            correct      += predicted.eq(labels_batch).sum().item()
            total        += labels_batch.size(0)
            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels_batch.cpu().numpy())

    return running_loss / total, 100. * correct / total, all_preds, all_labels


def train_model(model_name, model, num_epochs=NUM_EPOCHS):
    """
    Full training loop for one model.
    Returns history dict with train/val losses and accuracies.
    """
    print(f"\n{'─'*50}")
    print(f"🚀 Training: {model_name}")
    print(f"{'─'*50}")

    model = model.to(DEVICE)

    # Use class weights in the loss to handle imbalance
    class_counts  = [train_counts.get(i, 1) for i in range(NUM_CLASSES)]
    weights_tensor = torch.FloatTensor([1.0/c for c in class_counts]).to(DEVICE)
    criterion = nn.CrossEntropyLoss(weight=weights_tensor)

    # Adam optimizer — a good default
    optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()),
                           lr=LEARNING_RATE, weight_decay=1e-4)

    # Learning rate scheduler: reduce LR if val loss stalls
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.5, patience=3, 
    )

    history = {
        'train_loss': [], 'val_loss': [],
        'train_acc':  [], 'val_acc':  []
    }
    best_val_acc = 0.0

    for epoch in range(1, num_epochs + 1):
        train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer)
        val_loss,   val_acc, _, _ = evaluate(model, val_loader, criterion)

        scheduler.step(val_loss)

        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['train_acc'].append(train_acc)
        history['val_acc'].append(val_acc)

        # Save best model
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save(model.state_dict(), f'{OUTPUT_DIR}/best_{model_name}.pth')

        print(f"  Epoch {epoch:02d}/{num_epochs}  "
              f"Train Loss: {train_loss:.4f}  Train Acc: {train_acc:.2f}%  |  "
              f"Val Loss: {val_loss:.4f}  Val Acc: {val_acc:.2f}%"
              + (" ★ Best" if val_acc == best_val_acc else ""))

    # Load best weights before returning
    model.load_state_dict(torch.load(f'{OUTPUT_DIR}/best_{model_name}.pth',
                                      map_location=DEVICE))
    return model, history

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# SECTION 8 — POST-TRAINING VISUALIZATION FUNCTIONS
# ─────────────────────────────────────────────────────────────────────────────

def plot_training_curves(history, model_name, save_path):
    """VIZ: Loss and Accuracy curves over epochs."""
    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    fig.suptitle(f'Training History — {model_name}', fontsize=13, fontweight='bold')
    epochs = range(1, len(history['train_loss']) + 1)

    # Loss
    axes[0].plot(epochs, history['train_loss'], 'b-o', markersize=4, label='Train Loss')
    axes[0].plot(epochs, history['val_loss'],   'r-o', markersize=4, label='Val Loss')
    axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
    axes[0].set_title('Loss Curves')
    axes[0].legend(); axes[0].grid(alpha=0.3)

    # Accuracy
    axes[1].plot(epochs, history['train_acc'], 'b-o', markersize=4, label='Train Acc')
    axes[1].plot(epochs, history['val_acc'],   'r-o', markersize=4, label='Val Acc')
    axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Accuracy (%)')
    axes[1].set_title('Accuracy Curves')
    axes[1].legend(); axes[1].grid(alpha=0.3)
    axes[1].set_ylim([0, 105])

    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f"   💾 Saved: {save_path}")


def plot_confusion_matrix(y_true, y_pred, class_names, model_name, save_path):
    """VIZ: Confusion matrix heatmap."""
    cm = confusion_matrix(y_true, y_pred)
    cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]  # normalize

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle(f'Confusion Matrix — {model_name}', fontsize=13, fontweight='bold')

    short_names = [c.replace('[Malignant] ', 'M.') for c in class_names]

    # Raw counts
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=short_names, yticklabels=short_names, ax=axes[0])
    axes[0].set_xlabel('Predicted'); axes[0].set_ylabel('True')
    axes[0].set_title('Raw Counts')

    # Normalized
    sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Blues',
                xticklabels=short_names, yticklabels=short_names, ax=axes[1])
    axes[1].set_xlabel('Predicted'); axes[1].set_ylabel('True')
    axes[1].set_title('Normalized (row = true class)')

    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f"   💾 Saved: {save_path}")


def plot_sample_predictions(model, test_loader, class_names, model_name,
                            save_path, n_images=12):
    """VIZ: Show test images with predicted vs true labels."""
    model.eval()
    images_shown, preds_shown, labels_shown = [], [], []

    with torch.no_grad():
        for images, lbs in test_loader:
            outputs   = model(images.to(DEVICE))
            _, predicted = outputs.max(1)
            images_shown.extend(images[:4])
            preds_shown.extend(predicted.cpu()[:4].tolist())
            labels_shown.extend(lbs[:4].tolist())
            if len(images_shown) >= n_images:
                break

    # Denormalize for display
    inv_mean = [-m/s for m, s in zip(IMAGENET_MEAN, IMAGENET_STD)]
    inv_std  = [1/s for s in IMAGENET_STD]
    denorm   = transforms.Normalize(mean=inv_mean, std=inv_std)

    cols = 4
    rows = n_images // cols
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 3.5, rows * 3.5))
    fig.suptitle(f'Sample Predictions — {model_name}\n(Green=Correct, Red=Wrong)',
                 fontsize=12, fontweight='bold')

    short = [c.replace('[Malignant] ', 'M.') for c in class_names]

    for i in range(n_images):
        ax  = axes[i // cols][i % cols]
        img = denorm(images_shown[i]).permute(1, 2, 0).clamp(0, 1).numpy()
        ax.imshow(img)
        correct = preds_shown[i] == labels_shown[i]
        color   = 'green' if correct else 'red'
        ax.set_title(f"True: {short[labels_shown[i]]}\nPred: {short[preds_shown[i]]}",
                     fontsize=8, color=color, fontweight='bold')
        ax.axis('off')
        for spine in ax.spines.values():
            spine.set_edgecolor(color)
            spine.set_linewidth(3)

    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f"   💾 Saved: {save_path}")


def plot_all_models_comparison(results_dict, save_path):
    """VIZ: Final comparison bar chart of all 5 models."""
    model_names = list(results_dict.keys())
    test_accs   = [results_dict[m]['test_acc'] for m in model_names]
    val_accs    = [results_dict[m]['best_val_acc'] for m in model_names]

    x = np.arange(len(model_names))
    width = 0.35

    fig, ax = plt.subplots(figsize=(12, 6))
    bars1 = ax.bar(x - width/2, val_accs,  width, label='Best Val Acc',  color='#3498db', edgecolor='black')
    bars2 = ax.bar(x + width/2, test_accs, width, label='Test Acc',      color='#e74c3c', edgecolor='black')

    ax.set_ylabel('Accuracy (%)')
    ax.set_title('Model Comparison — Validation vs Test Accuracy', fontsize=13, fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels(model_names, rotation=15, ha='right')
    ax.legend()
    ax.set_ylim([0, 110])
    ax.grid(axis='y', alpha=0.3)
    ax.axhline(y=90, color='green', linestyle='--', alpha=0.5, label='90% line')

    for bar in bars1:
        ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.5,
                f'{bar.get_height():.1f}%', ha='center', va='bottom', fontsize=9, fontweight='bold')
    for bar in bars2:
        ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.5,
                f'{bar.get_height():.1f}%', ha='center', va='bottom', fontsize=9, fontweight='bold')

    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f"   💾 Saved: {save_path}")



In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# SECTION 9 — MAIN: TRAIN ALL 5 MODELS
# ─────────────────────────────────────────────────────────────────────────────

print("\n" + "="*60)
print("🏋️  SECTION 9: TRAINING ALL 5 MODELS")
print("="*60)

results = {}  # Stores test accuracy and history for each model

for model_name, builder in model_builders.items():
    set_seed(SEED)

    model = builder()
    model, history = train_model(model_name, model, num_epochs=NUM_EPOCHS)

    # Evaluate on test set
    _, _, test_preds, test_labels_list = evaluate(
        model, test_loader,
        nn.CrossEntropyLoss()
    )
    test_acc = 100. * sum(p == t for p, t in zip(test_preds, test_labels_list)) / len(test_labels_list)

    results[model_name] = {
        'history':      history,
        'test_acc':     test_acc,
        'best_val_acc': max(history['val_acc']),
        'test_preds':   test_preds,
        'test_labels':  test_labels_list,
    }

    # ── Post-training visualizations for this model ───────────────────────
    print(f"\n📊 Generating visualizations for {model_name}...")

    plot_training_curves(
        history, model_name,
        f'{OUTPUT_DIR}/viz_curves_{model_name}.png'
    )
    plot_confusion_matrix(
        test_labels_list, test_preds, CLASS_NAMES, model_name,
        f'{OUTPUT_DIR}/viz_confmat_{model_name}.png'
    )
    plot_sample_predictions(
        model, test_loader, CLASS_NAMES, model_name,
        f'{OUTPUT_DIR}/viz_preds_{model_name}.png'
    )

    print(f"\n✅ {model_name} Test Accuracy: {test_acc:.2f}%")
    print(f"\n{classification_report(test_labels_list, test_preds, target_names=CLASS_NAMES)}")

    # Free GPU memory before next model
    del model
    torch.cuda.empty_cache() if DEVICE.type == 'cuda' else None


In [ ]:

# ─────────────────────────────────────────────────────────────────────────────
# SECTION 10 — FINAL COMPARISON VISUALIZATION
# ─────────────────────────────────────────────────────────────────────────────

print("\n" + "="*60)
print("🏆 SECTION 10: FINAL COMPARISON")
print("="*60)

plot_all_models_comparison(results, f'{OUTPUT_DIR}/viz_final_comparison.png')

# ── Summary Table ─────────────────────────────────────────────────────────
print("\n📋 FINAL RESULTS SUMMARY:")
print(f"{'Model':<25} {'Best Val Acc':>14} {'Test Acc':>10}")
print("─"*52)
for name, r in results.items():
    print(f"{name:<25} {r['best_val_acc']:>13.2f}%  {r['test_acc']:>9.2f}%")

print("\n✅ All done! Check the 'outputs/' folder for all saved plots.")
print(f"   Total visualizations saved: {len(list(Path(OUTPUT_DIR).glob('*.png')))}")

In [ ]:
!pip install pennylane pennylane-lightning -q

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

import pennylane as qml
import numpy as np

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
import numpy as np
import pennylane as qml

################################################
# Data Preparation
################################################

def tf_to_numpy(dataset):
    images = []
    labels = []

    for x, y in dataset:
        images.append(x.numpy())
        labels.append(y)

    images = np.array(images)
    labels = np.array(labels)

    return images, labels

X_train_np, y_train_np = tf_to_numpy(train_dataset)
X_test_np, y_test_np   = tf_to_numpy(test_dataset)

# FIX: Removed the .permute() call! 
# The dataset is already [Batch, Channels, Height, Width]
X_train = torch.tensor(X_train_np).float()
X_test  = torch.tensor(X_test_np).float()

y_train = torch.tensor(y_train_np).float()
y_test  = torch.tensor(y_test_np).float()

################################################
# DataLoader
################################################

train_dataset_pt = TensorDataset(X_train, y_train)
test_dataset_pt  = TensorDataset(X_test, y_test)

trainloader = DataLoader(train_dataset_pt, batch_size=32, shuffle=True)
testloader  = DataLoader(test_dataset_pt, batch_size=32, shuffle=False)

################################################
# Quantum Circuit
################################################

n_qubits = 4
dev = qml.device("default.qubit", wires=n_qubits)

@qml.qnode(dev, interface="torch")
def quantum_circuit(inputs, weights):
    qml.AngleEmbedding(inputs, wires=range(n_qubits))
    qml.StronglyEntanglingLayers(weights, wires=range(n_qubits))
    return [qml.expval(qml.PauliZ(i)) for i in range(n_qubits)]

weight_shapes = {"weights": (3, n_qubits, 3)}
quantum_layer = qml.qnn.TorchLayer(quantum_circuit, weight_shapes)

################################################
# Hybrid Quantum CNN Model
################################################

class HybridQCNN(nn.Module):
    def __init__(self):
        super(HybridQCNN, self).__init__()
        
        self.conv = nn.Sequential(
            nn.Conv2d(3, 8, kernel_size=3),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.AdaptiveAvgPool2d((7,7))
        )
        
        self.flatten = nn.Flatten()
        self.fc1 = nn.Linear(8 * 7 * 7, 4)
        self.quantum = quantum_layer
        self.fc2 = nn.Linear(4, 1)

    def forward(self, x):
        x = self.conv(x)
        x = self.flatten(x)
        x = self.fc1(x)
        x = torch.tanh(x)
        x = self.quantum(x)
        x = self.fc2(x)
        return torch.sigmoid(x)

model = HybridQCNN()

################################################
# Training Setup
################################################

criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

################################################
# Training Loop
################################################

################################################
# CLEAN & BULLETPROOF TRAINING LOOP
################################################

for epoch in range(5):
    running_loss = 0.0

    for images, labels in trainloader:
        
        # --- 1. THE CHANNEL FIX ---
        # Forces images into [Batch, Channels, Height, Width] regardless of current state
        if len(images.shape) == 4:
            if images.shape[3] == 3:
                images = images.permute(0, 3, 1, 2)
            elif images.shape[2] == 3:
                images = images.permute(0, 2, 1, 3)

        # --- 2. THE LABEL FIX ---
        # Add the required dimension for BCELoss
        labels = labels.unsqueeze(1).float()
        
        # The Ultimate Squash: Maps any number > 0 to 1.0, and anything <= 0 to 0.0.
        # This instantly fixes datasets with [-1, 1], [0, 255], or weird float values.
        labels = torch.where(labels > 0, 1.0, 0.0)

        # --- 3. THE FORWARD & BACKWARD PASS ---
        optimizer.zero_grad()
        
        outputs = model(images)
        loss = criterion(outputs, labels)
        
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        
    print(f"Epoch {epoch+1}, Loss: {running_loss/len(trainloader):.4f}")

In [ ]:
################################################
# Testing
################################################

correct = 0
total = 0

# Set model to evaluation mode (good practice!)
model.eval()

with torch.no_grad():
    for images, labels in testloader:
        
        # --- 1. THE CHANNEL FIX ---
        if len(images.shape) == 4:
            if images.shape[3] == 3:
                images = images.permute(0, 3, 1, 2)
            elif images.shape[2] == 3:
                images = images.permute(0, 2, 1, 3)

        # --- 2. THE LABEL FIX ---
        labels = labels.unsqueeze(1).float()
        labels = torch.where(labels > 0, 1.0, 0.0)

        # --- 3. FORWARD PASS ---
        outputs = model(images)
        
        # If output is > 0.5, predict 1.0. Otherwise predict 0.0
        predicted = (outputs > 0.5).float()
        
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

accuracy = 100 * correct / total
print(f"Test Accuracy: {accuracy:.2f}%")

# Set model back to training mode just in case you run the training cell again
model.train()

In [ ]:
import torch
import matplotlib.pyplot as plt

# --- 1. CREATE DUMMY DATA ---
# Create a random tensor of 4 values for the 4 qubits
sample_inputs = torch.rand(4)

# Create a random tensor matching your (3, n_qubits, 3) weight shape
sample_weights = torch.rand(3, 4, 3)

# --- 2. DRAW THE CIRCUIT ---
fig, ax = qml.draw_mpl(quantum_circuit)(sample_inputs, sample_weights)
plt.show()

# Save the figure
fig.savefig('quantum_circuit.png', dpi=300, bbox_inches='tight')

In [ ]:
print(qml.draw(quantum_circuit)(torch.randn(4), torch.randn(3,4,3)))